# Damages With Road Assets (Illustrative Case)


This notebook reproduces the workflow from `examples/sito_2025_asset_based_damage_examples/illustrative_case.py`.

It builds a polygon from the hazard raster extent, configures network and hazard data, and runs an asset-based damages analysis.

## Imports

In [ ]:
from pathlib import Path


import geopandas as gpd
import pyproj
import rasterio
from shapely.geometry import box as make_box
from shapely.ops import transform


from ra2ce.ra2ce_handler import Ra2ceHandler


from ra2ce.network.network_config_data.network_config_data import (
    HazardSection,
    NetworkConfigData,
    NetworkSection,
 )
from ra2ce.network.network_config_data.enums.aggregate_wl_enum import AggregateWlEnum
from ra2ce.network.network_config_data.enums.network_type_enum import NetworkTypeEnum
from ra2ce.network.network_config_data.enums.road_type_enum import RoadTypeEnum
from ra2ce.network.network_config_data.enums.source_enum import SourceEnum


from ra2ce.analysis.damages.damages import AnalysisSectionDamages
from ra2ce.analysis.analysis_config_data.analysis_config_data import AnalysisConfigData
from ra2ce.analysis.analysis_config_data.enums.analysis_damages_enum import AnalysisDamagesEnum
from ra2ce.analysis.analysis_config_data.enums.damage_curve_enum import DamageCurveEnum
from ra2ce.analysis.analysis_config_data.enums.event_type_enum import EventTypeEnum
from ra2ce.analysis.analysis_config_data.enums.risk_calculation_mode_enum import RiskCalculationModeEnum

## Paths And Input Data

In [ ]:
root_dir = Path("illustrative_case")
assert root_dir.exists(), "root_dir not found."


static_path = root_dir / "static"
hazard_path = root_dir / "hazard"
output_path = root_dir / "output"


hazard_map = list(hazard_path.glob("*.tif"))
assert hazard_map, "No hazard raster found in illustrative_case/hazard"


print(f"Root: {root_dir.resolve()}")
print(f"Hazard files: {len(hazard_map)}")
for p in hazard_map:
    print(f" - {p.name}")

## Build Study Area Polygon From Hazard Extent

In [ ]:
def reproject_geometry(geom, src_crs, dst_crs):
    project = pyproj.Transformer.from_crs(src_crs, dst_crs, always_xy=True).transform
    return transform(project, geom)


with rasterio.open(hazard_map[0]) as src:
    bbox = src.bounds
    bbox_polygon = make_box(bbox.left, bbox.bottom, bbox.right, bbox.top)
    src_crs = src.crs
    dst_crs = 4326


    if src_crs.to_epsg() != dst_crs:
        bbox_polygon = reproject_geometry(
            bbox_polygon,
            src_crs,
            pyproj.CRS.from_epsg(dst_crs),
        )
        print(f"Hazard map CRS: {src_crs}. Reprojected polygon to EPSG:{dst_crs}.")


gdf_polygon = gpd.GeoDataFrame(index=[0], geometry=[bbox_polygon], crs=dst_crs)
polygon_fp = static_path / "polygon.geojson"
gdf_polygon.to_file(polygon_fp, driver="GeoJSON")
print(f"Saved study area polygon: {polygon_fp}")

## Configure Network And Hazard

In [ ]:
network_section = NetworkSection(
    network_type=NetworkTypeEnum.DRIVE,
    source=SourceEnum.OSM_DOWNLOAD,
    polygon=polygon_fp,
    save_gpkg=True,
    road_types=[
        RoadTypeEnum.TERTIARY,
        RoadTypeEnum.TERTIARY_LINK,
        RoadTypeEnum.SECONDARY,
        RoadTypeEnum.SECONDARY_LINK,
        RoadTypeEnum.PRIMARY,
        RoadTypeEnum.PRIMARY_LINK,
        RoadTypeEnum.TRUNK,
        RoadTypeEnum.MOTORWAY,
        RoadTypeEnum.MOTORWAY_LINK,
    ],
    reuse_network_output=True,
 )


hazard_section = HazardSection(
    hazard_map=[Path(file) for file in hazard_path.glob("*.tif")],
    aggregate_wl=AggregateWlEnum.MEAN,
    hazard_crs="EPSG:4326",
 )


network_config_data = NetworkConfigData(
    root_path=root_dir,
    static_path=static_path,
    network=network_section,
    hazard=hazard_section,
 )
network_config_data.network.save_gpkg = True


network_config_data

## Configure Damages Analysis

In [ ]:
damages_analysis = [
    AnalysisSectionDamages(
        name="damages_with_asset",
        analysis=AnalysisDamagesEnum.DAMAGES_WITH_ASSETS,
        event_type=EventTypeEnum.RETURN_PERIOD,
        damage_curve=DamageCurveEnum.MAN,
        risk_calculation_mode=RiskCalculationModeEnum.TRIANGLE_TO_NULL_YEAR,
        risk_calculation_year=5,
        save_csv=True,
        save_gpkg=True,
    )
]


analysis_config_data = AnalysisConfigData(
    analyses=damages_analysis,
    root_path=root_dir,
    output_path=output_path,
 )
analysis_config_data.input_path = root_dir / "input_data"


analysis_config_data

## Run RA2CE

In [ ]:
Ra2ceHandler.run_with_config_data(network_config_data, analysis_config_data)

## Inspect Outputs

## Inspect Outputs

In [ ]:
if output_path.exists():
    output_files = sorted(output_path.rglob("*"))
    print(f"Output files found: {len(output_files)}")
    for fp in output_files[:40]:
        print(fp.relative_to(root_dir))
else:
    print("Output folder does not exist yet. Run the previous cell first.")